In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

# Ames Housing Price Prediction
**Author:** Sai Chaithanyia

## Problem Statement
The Ames Housing dataset contains 80+ features describing residential homes in Ames, Iowa. 
The goal is to build a regression model that accurately predicts sale prices, 
by applying systematic preprocessing, feature engineering, and model comparison.

# IMPORTING THE DATASET AND DOING INITIAL ANALYSIS

Performing basic feature exploration after loading the dataset.

In [ ]:
data=pd.read_csv("data/AmesHousing.csv")
data.head()

In [ ]:
data.head()

In [ ]:
data.shape

In [ ]:
data.tail()

In [ ]:
data.columns

In [ ]:
data.dtypes

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
num_cols = data.select_dtypes(include=np.number).columns.drop('SalePrice')
cat_cols=data.select_dtypes(include=object).columns

In [ ]:
isna_cols=data.isna().sum()
isna_indexes=[i for i in range(len(isna_cols)) if isna_cols[i]!=0]
isna_cols=[(data.columns[i],data[data.columns[i]].dtype) for i in isna_indexes]
isna_cols

In [ ]:
nan_num_cols=list(set(num_cols)&(set(list(map(lambda x:x[0],isna_cols)))))
nan_num_cols

In [ ]:
fig,axes=plt.subplots(6,2,figsize=(12,17))
axes=axes.flatten()
for i,col in enumerate(nan_num_cols):
    sns.histplot(data[col],kde=True,color='purple',ax=axes[i])

In [ ]:
#using median for all except Total Bsmt SF and Garage Area(mean)
data['Garage Area']=data['Garage Area'].fillna(data['Garage Area'].mean())
data['Total Bsmt SF']=data['Total Bsmt SF'].fillna(data['Total Bsmt SF'].mean())

for col in nan_num_cols:
    if col!='Total Bsmt SF' and col!='Garage Area':
        data[col]=data[col].fillna(data[col].median())

In [ ]:
for c in cat_cols:
    print(data[c].value_counts())

In [ ]:
num_cols

# Column datatype manipulation

In [ ]:
#Using None or Mode for Categorical
nones=['Pool QC', 'Fence', 'Misc Feature','Alley','Fireplace Qu', 'Garage Type', 'Garage Finish', 'Garage Qual',
       'Garage Cond','Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
       'BsmtFin Type 1', 'BsmtFin Type 2']
modes=list(set(cat_cols)-(set(nones)))

for col in nones:
    data[col]=data[col].fillna("None")
for col in modes:
    data[col]=data[col].fillna(data[col].mode()[0])

In [ ]:
column_na=[]
for col in cat_cols:
    if data[col].isna().sum()!=0:
        column_na.append(col)
column_na
#column_na empty means there is no NaN values

In [ ]:
sum(list(data.isna().sum()))

In [ ]:
qual_map = {
    "None": 0,
    "Po": 1,
    "Fa": 2,
    "TA": 3,
    "Gd": 4,
    "Ex": 5
}
ordinal_cols = [
    'Exter Qual','Exter Cond','Bsmt Qual','Bsmt Cond',
    'Heating QC','Kitchen Qual','Fireplace Qu',
    'Garage Qual','Garage Cond','Pool QC'
]

for col in ordinal_cols:
    data[col] = data[col].map(qual_map)

In [ ]:
data['MS SubClass'] = data['MS SubClass'].astype(str)

In [ ]:
data['Yr Sold'] = data['Yr Sold'].astype(str)
data['Mo Sold']  = data['Mo Sold'].astype(str)
data['Garage Yr Blt']=data['Garage Yr Blt'].astype(str)

In [ ]:
data['Year Built']=data['Year Built'] .astype(str)
data['Garage Cars'] = data['Garage Cars'].astype(str)
data['Year Remod/Add']=data['Year Remod/Add'].astype(str)

order and pid columns are unecessary in prediction so dropping them

In [ ]:
data.drop(['Order','PID'],axis=1,inplace=True)


In [ ]:
y=data['SalePrice']
x=data.drop('SalePrice',axis=1)

In [ ]:
num_cols=data.select_dtypes(include=np.number).columns.drop('SalePrice')
cat_cols=data.select_dtypes(include=object).columns

# Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,Y_train,Y_test=train_test_split(x,y,test_size=0.2,random_state=33)

X_train.shape

# *PREPOCESSING ON TRAIN DATA*


# Transformation

In [ ]:
X_train[num_cols].skew()

In [ ]:
data[num_cols].kurt()

In [ ]:
heavy_skewed_cols=data[num_cols].skew()[abs(data[num_cols].skew())>1].index.tolist()
len(heavy_skewed_cols)

In [ ]:
neg=[]
zero=[]
for x in heavy_skewed_cols:
    if (X_train[x]==0).any()==True:
        zero.append(x)
    if (X_train[x]<0).any()==True:
        neg.append(x)
zero
#log1p needed

In [ ]:
neg
#No yeo Jhonson

In [ ]:
pos=list(set(heavy_skewed_cols)-set(zero))
pos
#Boxcox needed

LOG TRANSFORMATION

In [ ]:
X_train[zero]=np.log1p(X_train[zero])
X_train[zero]

BOXOCOX TRANSFORMATION

In [ ]:
from scipy.stats import boxcox
lambdaval={}
for col in pos:
    X_train[col],lambdaval[col]=boxcox(X_train[col])
X_train[pos]

Since the target in traning dataset is highly right skewed so applying log transformation

In [ ]:
Y_train.skew()
Y_train=np.log1p(Y_train)

# Feature Selection

In [ ]:
X_train[num_cols]

**Feature selection with pearson correlation for numerical columns**

In [ ]:
corr=X_train[num_cols].corr()
corr

In [ ]:
num_data=X_train[num_cols]
non_imp_feat=[]
threshold=0.8
for i in range(len(corr)):
    for j in range(i):
        if abs(corr.iloc[i,j])>=threshold:
            non_imp_feat.append(num_cols[i])
non_imp_feat

In [ ]:
num_data.drop(non_imp_feat,axis=1,inplace=True)

In [ ]:
num_data.shape

In [ ]:
num_data.columns

**Cardinality Analysis**

In [ ]:
cardinal_vals=pd.DataFrame(columns=['column_name','unique_count_entries'])
cat_data=X_train[cat_cols]
for col in cat_data.columns:
    cardinal_vals=pd.concat([cardinal_vals,pd.DataFrame([{'column_name': col, 'unique_count_entries': cat_data[col].nunique()}])])
cardinal_vals

# Encoding Catergorical Columns

**One Hot Encoding for low cardinality**

**Frequency Encoding for High cardinality**

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe=OneHotEncoder(sparse_output=False,handle_unknown='ignore')
ohe_cols=[col for col in cat_cols if X_train[col].nunique()<15]
cat_data_enc=ohe.fit_transform(X_train[ohe_cols])
encoded_cols=ohe.get_feature_names_out()
cat_data_enc=pd.DataFrame(cat_data_enc,columns=encoded_cols,index=X_train[ohe_cols].index)
cat_data_enc

In [ ]:
high_card_cols=[col for col in cat_cols if col not in ohe_cols]

In [ ]:
X_train_encoded=pd.DataFrame(index=X_train.index)
freq_map={}
for col in high_card_cols:
    freq_map[col]=X_train[col].value_counts(normalize=True)
    X_train_encoded[col]=X_train[col].map(freq_map[col])
X_train_encoded
    

In [ ]:
cat_data=pd.concat([cat_data_enc,X_train_encoded],axis=1)
cat_data

**Selecting Features with pearson correlation for categorical columns**

In [ ]:
cat_corr=pd.Series(cat_data.corrwith(Y_train))
imp=[]
for index,corr in cat_corr.items():
    if abs(corr)>0.02:
        imp.append(index)
non_imp=set(cat_data.columns)-set(imp)
cat_data.drop(non_imp,axis=1,inplace=True)

In [ ]:
cat_data.shape

# Scaling

**Using Robust Scaler because of huge outliers in the dataset**

In [ ]:
from sklearn.preprocessing import RobustScaler
rs=RobustScaler()
num_data_enc1=rs.fit_transform(num_data)
num_data_enc1=pd.DataFrame(num_data_enc1,columns=num_data.columns,index=num_data.index)
num_data_enc1

In [ ]:
X_train_final=pd.concat([cat_data,num_data_enc1],axis=1)
X_train_final.head()

# *PREPROCESSING ON TEST DATA*

The lambdaval is a dictionary that stores all the lambda values used in the training data with only positive values during boxcox transformation and these values should again be used in test data.

In [ ]:
lambdaval

BOXCOX TRANSFORM

In [ ]:
from scipy.special import boxcox1p
for col in pos:
    X_test[col]=boxcox1p(X_test[col],lambdaval[col])
X_test

LOG TRANSFORM

In [ ]:
for col in zero:
    X_test[col]=np.log1p(X_test[col])
X_test

In [ ]:
cat_test=X_test[cat_cols]
cat_test

**ONE HOT ENCODING OF CATEGORICAL COLUMNS**

In [ ]:
X_test_enc=ohe.transform(cat_test[ohe_cols])
X_test_enc=pd.DataFrame(X_test_enc,columns=encoded_cols,index=cat_test.index)
X_test_enc

**FREQUENCY ENCODING COLUMNS**

In [ ]:
X_test_encoded = pd.DataFrame(index=X_test.index)
for col in high_card_cols:
    X_test_encoded[col] = X_test[col].map(freq_map[col]).fillna(0)
X_test_encoded.isna().sum()

In [ ]:
X_test_cat=pd.concat([X_test_enc,X_test_encoded],axis=1)

Dropping non important columns from test data that are derived from training data

In [ ]:
num_test=X_test[num_cols]
num_test

In [ ]:
num_test.drop(non_imp_feat,axis=1,inplace=True)

In [ ]:
num_test.columns

In [ ]:
X_test_cat.drop(non_imp,axis=1,inplace=True)

**Scaling with same robust scaler used in training data**

In [ ]:
print(num_data.columns.equals(num_test.columns))

In [ ]:
num_test_scaled=rs.transform(num_test)
num_test_scaled=pd.DataFrame(num_test_scaled,columns=num_test.columns,index=num_test.index)
num_test_scaled

**Applying log transform on test target due to high right skewness**

In [ ]:
Y_test=np.log1p(Y_test)

In [ ]:
X_test_final=pd.concat([X_test_cat,num_test_scaled],axis=1)

In [ ]:
X_test_final = X_test_final.reindex(columns=X_train_final.columns, fill_value=0)


# APPLYING PCA 

PCA is applied to reduce the huge dimensionality

In [ ]:
from sklearn.decomposition import PCA

pca=PCA(n_components=0.95)
X_train_reduced=pca.fit_transform(X_train_final)

In [ ]:
print(X_train_final.shape)
X_train_reduced.shape

In [ ]:
X_test_reduced=pca.transform(X_test_final)
print(X_test_final.shape)
X_test_reduced.shape

In [ ]:
loadings = pd.DataFrame(pca.components_, columns=X_train_final.columns)
len(loadings.columns)
pc_columns = []
for i, row in loadings.iterrows():
    top_feature = row.abs().idxmax()  # feature with highest absolute loading
    pc_columns.append(f"{top_feature}_pc{i+1}")

final train data after preprocessing and PCA

In [ ]:
X_train_reduced_df=pd.DataFrame(X_train_reduced,columns=pc_columns,index=X_train_final.index)
X_train_reduced_df

In [ ]:
X_test_reduced=pca.transform(X_test_final)
print(X_test_final.shape)
X_test_reduced.shape

final test data after preprocessing and PCA

In [ ]:
X_test_reduced_df=pd.DataFrame(X_test_reduced,columns=X_train_reduced_df.columns,index=X_test_final.index)
X_test_reduced_df

Variances of top selected features

In [ ]:
explained_var=pca.explained_variance_ratio_
explained_var

Checking if top variances sum to a total of 0.95(as chosen)

In [ ]:
explained_var.sum()

Plotting the cumulative variances with number of components to see their effects on the dataset

In [ ]:
plt.plot(range(1, len(explained_var)+1), explained_var.cumsum(), marker='*')
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA - Cumulative Variance")
plt.show()

# MODEL TRAINING AND TUNING WITH LINEAR REGRESSION

Since all the features are scaled and normalised all the metrics are in the log scale

**PCA USED LINEAR REGRESSION**

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error,r2_score,mean_absolute_error
from sklearn.model_selection import GridSearchCV

lr=LinearRegression()
params={'fit_intercept':[True,False],'positive':[True, False]}
grid=GridSearchCV(estimator=lr,
                  param_grid=params,
                  cv=5,
                  scoring='neg_root_mean_squared_error',
                  n_jobs=-1)

grid.fit(X_train_reduced_df,Y_train)

best_model = grid.best_estimator_
print("Best Parameters :", grid.best_params_)

train_columns = X_train_reduced_df.columns

y_pred=best_model.predict(X_test_reduced_df)

rmse1=root_mean_squared_error(Y_test,y_pred)
mae1 = mean_absolute_error(Y_test, y_pred)                 
r21= r2_score(Y_test, y_pred)   
print('Root mean squared error \n:',rmse1)
print(f'r2 score: {r21}')
print(f'Mean absolute error: {mae1}')

**NO PCA LINEAR REGRESSION**

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error,r2_score,mean_absolute_error
from sklearn.model_selection import GridSearchCV

lr=LinearRegression()
params={'fit_intercept':[True,False],'positive':[True, False]}
grid=GridSearchCV(estimator=lr,
                  param_grid=params,
                  cv=5,
                  scoring='neg_root_mean_squared_error',
                  n_jobs=-1)

grid.fit(X_train_final,Y_train)

best_model = grid.best_estimator_
print("Best Parameters :", grid.best_params_)

train_columns = X_train_final.columns

demo_y_pred=best_model.predict(X_test_final)

demo_rmse1=root_mean_squared_error(Y_test,demo_y_pred)
demo_mae1 = mean_absolute_error(Y_test, demo_y_pred)                 
demo_r21= r2_score(Y_test, demo_y_pred)   
print('Root mean squared error \n:',demo_rmse1)
print(f'r2 score: {demo_r21}')
print(f'Mean absolute error: {demo_mae1}')

So it is very evident PCA outperforms normal linear regression because it explicitly handles correlated features.

actual test mean vs predicted values mean

In [ ]:
print(f"Actual Y_test mean: {Y_test.mean():.2f}")
print(f"Predicted Y mean: {y_pred.mean():.2f}")

Conversion from log to original scale and checking actual vs predicted means

In [ ]:
Y_test_og=np.expm1(Y_test)
Y_pred_og1=np.expm1(y_pred)

In [ ]:
print(f"Actual Y_test mean: {Y_test_og.mean():.2f}")
print(f"Predicted Y mean: {Y_pred_og1.mean():.2f}")

**MODEL EVALUATION METRICS ON ORIGINAL SCALE**

In [ ]:
rmse1_og=root_mean_squared_error(Y_test_og,Y_pred_og1)
mae1_og = mean_absolute_error(Y_test_og,Y_pred_og1)                 # MAE
r21_og= r2_score(Y_test_og,Y_pred_og1)   
print('Root mean squared error \n:',rmse1_og)
print(f'r2 score: {r21_og}')
print(f'Mean absolute error: {mae1_og}')

**PLOTTING ACTUAL VS PREDICTED**

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(Y_test, y_pred, alpha=0.5)
plt.plot([Y_test.min(),Y_test.max()],[y_pred.min(),y_pred.max()],linestyle="--")
plt.ylabel("Predicted (log SalePrice)")
plt.title("Predicted vs Actual (log scale)")
plt.show()

Predicted values closely follow actual values, indicating the model captures the overall trend well.

Small deviations around the diagonal line suggest low error and good predictive performance.

**PLOTTING RESIDUALS VS PREDICTED**

In [ ]:
residual=Y_test-y_pred

plt.scatter(y_pred,residual,alpha=0.6,color='red')
plt.axhline(0,linestyle='--',color='red')
plt.xlabel("Predicted values")
plt.ylabel("Residuals")
plt.title("Residuals vs Predictions")
plt.show()

Since the plots are scattered and clustered around X=0, hence the model captures all the features very well and there arent any non linear relationships left uncaptured by the model.

**PLOTTING DISTRIBUTION OF ACTUAL VS PREDICTED**

In [ ]:
plt.hist(Y_test,bins=50,color='green',alpha=0.4,label='Actual')
plt.hist(y_pred,bins=50,color='yellow',alpha=0.4,label='Predicted')
plt.legend()
plt.title("Distribution: Actual vs Predicted")
plt.show()

Both the distributions overlap each other without any skew thus the predictions are quite accurate

Predicted values closely follow actual values, indicating the model captures the overall trend well.

Small deviations around the diagonal line suggest low error and good predictive performance.

Both the distributions overlap each other without any skew thus the predictions are quite accurate

# MODEL TRAINING AND TUNING WITH POLYNOMIAL FEATURES BASED LINEAR REGRESSION

Interaction terms (A×B) are added to capture non-linear relationships between features before fitting a linear model.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2)

X_train_poly = poly.fit_transform(X_train_reduced_df)
X_test_poly  = poly.transform(X_test_reduced_df)

print(f'Original features  : {X_train_final.shape[1]}')
print(f'After polynomial   : {X_train_poly.shape[1]}')

In [ ]:
"""from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error,r2_score,mean_absolute_error
from sklearn.model_selection import GridSearchCV

lr=LinearRegression()
params={'fit_intercept':[True,False],'positive':[True, False]}
grid=GridSearchCV(estimator=lr,
                  param_grid=params,
                  cv=5,
                  scoring='neg_root_mean_squared_error',
                  n_jobs=-1)

grid.fit(X_train_poly,Y_train)

best_model = grid.best_estimator_
print("Best Parameters :", grid.best_params_)

poly_pred=best_model.predict(X_test_poly)

poly_rmse1=root_mean_squared_error(Y_test,poly_pred)
poly_mae1 = mean_absolute_error(Y_test, poly_pred)                
poly_r21= r2_score(Y_test, poly_pred)   
print('Root mean squared error \n:',poly_rmse1)
print(f'r2 score: {poly_r21}')
print(f'Mean absolute error: {poly_mae1}')"""

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error,r2_score,mean_absolute_error

lr=LinearRegression(fit_intercept=True,positive=True)

lr.fit(X_train_poly,Y_train)

poly_pred=lr.predict(X_test_poly)

poly_rmse1=root_mean_squared_error(Y_test,poly_pred)
poly_mae1 = mean_absolute_error(Y_test, poly_pred)                
poly_r21= r2_score(Y_test, poly_pred)   
print('Root mean squared error \n:',poly_rmse1)
print(f'r2 score: {poly_r21}')
print(f'Mean absolute error: {poly_mae1}')

# MODEL TRAINING AND TUNING WITH L2 REGULARISED REGRESSION {RIDGE}

Since all the features are scaled and normalised all the metrics are in the log scale

In [ ]:
from sklearn.linear_model import Ridge

r=Ridge(max_iter=10000,solver='auto',random_state=33)

params={
    'alpha': np.logspace(-4,2,20),  
    'fit_intercept': [True, False],
    'positive': [True, False]
}
grid=GridSearchCV(estimator=r,
                  param_grid=params,
                  cv=6,scoring='neg_root_mean_squared_error',
                  n_jobs=-1)
grid.fit(X_train_reduced_df,Y_train)

best_model=grid.best_estimator_

print('Best Parameters :',grid.best_params_)

y_pred2=best_model.predict(X_test_reduced_df)

rmse2 = root_mean_squared_error(Y_test, y_pred2)
mae2 = mean_absolute_error(Y_test, y_pred2)
r22 = r2_score(Y_test, y_pred2)

print('Root mean squared error:', rmse2)
print('Mean absolute error:', mae2)
print('R2 score:', r22)



Checking Actual vs Predicted means

In [ ]:
print(f"Actual Y_test mean: {Y_test.mean():.2f}")
print(f"Predicted Y mean: {y_pred2.mean():.2f}")

Conversion to original scale checking actual vs predicted means

In [ ]:
Y_pred_og2=np.expm1(y_pred2)

In [ ]:
print(f"Actual Y_test mean: {Y_test_og.mean():.2f}")
print(f"Predicted Y mean: {Y_pred_og2.mean():.2f}")

**Model Evaluation in original scale**

In [ ]:
rmse2_og=root_mean_squared_error(Y_test_og,Y_pred_og2)
mae2_og= mean_absolute_error(Y_test_og,Y_pred_og2)                
r22_og = r2_score(Y_test_og,Y_pred_og2)   
print('Root mean squared error \n:',rmse2_og)
print(f'r2 score: {r22_og}')
print(f'Mean absolute error: {mae2_og}')

**PLOTTING ACTUAL VS PREDICTED**

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(Y_test, y_pred2, alpha=0.5)
plt.plot([Y_test.min(),Y_test.max()],[y_pred2.min(),y_pred2.max()],linestyle="--")
plt.ylabel("Predicted (log SalePrice)")
plt.title("Predicted vs Actual (log scale)")
plt.show()

Predicted values closely follow actual values, indicating the model captures the overall trend well.

Small deviations around the diagonal line suggest low error and good predictive performance.

**PLOTTING RESIDUALS VS PREDICTED**

In [ ]:
residual=Y_test-y_pred2

plt.scatter(y_pred2,residual,alpha=0.6,color='red')
plt.axhline(0,linestyle='--',color='red')
plt.xlabel("Predicted values")
plt.ylabel("Residuals")
plt.title("Residuals vs Predictions")
plt.show()

Since the plots are scattered and clustered around X=0, hence the model captures all the features very well and there arent any non linear relationships left uncaptured by the model.

**PLOTTING DISTRIBUTIONS OF ACTUAL AND PREDICTED VALUES**

In [ ]:
plt.hist(Y_test,bins=50,color='green',alpha=0.4,label='Actual')
plt.hist(y_pred2,bins=50,color='yellow',alpha=0.4,label='Predicted')
plt.legend()
plt.title("Distribution: Actual vs Predicted")
plt.show()

Both the distributions overlap each other without any skew thus the predictions are quite accurate

**So, Linear models with PCA perform comparably to without PCA, validating that 95% variance retention was sufficient to preserve the predictive signal while effectively resolving multicollinearity.**

# MODEL TRAINING AND TUNING WITH XGBOOST {Extreme Gradient Boosting}

Tree-based model that handles non-linear relationships natively.
Does not require skew transformation or PCA — passed directly from preprocessed data.
Alpha tuned via GridSearchCV across n_estimators, max_depth, learning_rate, and subsample.

PCA was applied for linear models to address multicollinearity and reduce dimensionality. However, since XGBoost is a tree-based ensemble that handles correlated features natively and splits on individual features, applying PCA offers no structural benefit and introduces information loss. Testing confirmed that XGBoost trained on the full preprocessed feature set outperformed the PCA-reduced version, so the final XGBoost model bypasses PCA.

Marking down the below Hyperparametric tuning with GridSearchCV due to heavy overload in time.

In [ ]:
"""from xgboost import XGBRegressor

grid_xgb = GridSearchCV(
    estimator=XGBRegressor(
        objective='reg:squarederror',
        random_state=33,tree_method='hist',
        n_jobs=-1
    ),
    param_grid={
        'n_estimators':  [500,600],
        'max_depth':     [5,4],
        'learning_rate': [0.05, 0.1],
        'subsample':     [0.7, 0.8],
        'reg_alpha':[1,0],
        'reg_lambda':[0,1],
        'colsample_bytree':[0.7,0.8,0.9]
    },
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

grid_xgb.fit(X_train_final,Y_train)

print('\nBest Parameters:', grid_xgb.best_params_)"""

**Output:**
Best Parameters: {'colsample_bytree': 0.9, 'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 600, 'reg_alpha': 0, 'reg_lambda': 1, 'subsample': 0.7}

**XGBOOST WITHOUT PCA**

In [ ]:
from xgboost import XGBRegressor

xgb=XGBRegressor(
    objective='reg:squarederror',
    tree_method='hist',
    colsample_bytree=0.9,
    learning_rate=0.05,
    max_depth=4,
    n_estimators=600,
    reg_alpha=0,
    reg_lambda=1,
    subsample=0.7,
    random_state=33,
    n_jobs=-1
)
xgb.fit(X_train_final,Y_train)
y_pred3=xgb.predict(X_test_final)

rmse3 = root_mean_squared_error(Y_test, y_pred3)
mae3 = mean_absolute_error(Y_test, y_pred3)
r23 = r2_score(Y_test, y_pred3)

print('Root mean squared error:', rmse3)
print('Mean absolute error:', mae3)
print('R2 score:', r23)


**XGBOOST WITH PCA**

In [ ]:
from xgboost import XGBRegressor

xgb1=XGBRegressor(
    objective='reg:squarederror',
    tree_method='hist',
    colsample_bytree=0.9,
    learning_rate=0.05,
    max_depth=4,
    n_estimators=600,
    reg_alpha=0,
    reg_lambda=1,
    subsample=0.7,
    random_state=33,
    n_jobs=-1
)
xgb1.fit(X_train_reduced_df,Y_train)
demo_y_pred3=xgb.predict(X_test_reduced_df)

demo_rmse3 = root_mean_squared_error(Y_test, demo_y_pred3)
demo_mae3 = mean_absolute_error(Y_test, demo_y_pred3)
demo_r23 = r2_score(Y_test, demo_y_pred3)

print('Root mean squared error:', demo_rmse3)
print('Mean absolute error:', demo_mae3)
print('R2 score:', demo_r23)


It is clearly evident that there is a little improvement in the scores without PCA. So skipping XGB with PCA for further analysis

In [ ]:
xgb.feature_importances_

Identifying the importance of features in the model, and plotting and finding top 10 most important features that dominates prediction.

In [ ]:
pd.Series(xgb.feature_importances_, index=X_train_final.columns)\
  .nlargest(10).plot(kind='barh', title='Top 10 Feature Importances')

Checking actual vs predicted means

In [ ]:
print(f"Actual Y_test mean: {Y_test.mean():.2f}")
print(f"Predicted Y mean: {y_pred3.mean():.2f}")

Conversion to original scale checking actual vs predicted means

In [ ]:
Y_pred_og3=np.expm1(y_pred3)

In [ ]:
print(f"Actual Y_test mean: {Y_test_og.mean():.2f}")
print(f"Predicted Y mean: {Y_pred_og3.mean():.2f}")

Model Evaluation in original scale

In [ ]:
rmse3_og=root_mean_squared_error(Y_test_og,Y_pred_og3)
mae3_og= mean_absolute_error(Y_test_og,Y_pred_og3)                
r23_og = r2_score(Y_test_og,Y_pred_og3)   
print('Root mean squared error \n:',rmse3_og)
print(f'r2 score: {r23_og}')
print(f'Mean absolute error: {mae3_og}')

**PLOTTING ACTUAL VS PREDICTED**

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(Y_test, y_pred3, alpha=0.5)
plt.plot([Y_test.min(),Y_test.max()],[y_pred3.min(),y_pred3.max()],linestyle="--")
plt.ylabel("Predicted (log SalePrice)")
plt.title("Predicted vs Actual (log scale)")
plt.show()

**PLOTTING RESIDUALS VS PREDICTED**

In [ ]:
residual=Y_test-y_pred3

plt.scatter(y_pred3,residual,alpha=0.6,color='red')
plt.axhline(0,linestyle='--',color='red')
plt.xlabel("Predicted values")
plt.ylabel("Residuals")
plt.title("Residuals vs Predictions")
plt.show()

**PLOTTING DISTRIBUTIONS OF ACTUAL AND PREDICTED VALUES**

In [ ]:
plt.hist(Y_test,bins=50,color='green',alpha=0.4,label='Actual')
plt.hist(y_pred3,bins=50,color='yellow',alpha=0.4,label='Predicted')
plt.legend()
plt.title("Distribution: Actual vs Predicted")
plt.show()

Both the distributions overlap each other without any skew thus the predictions are quite accurate

# COMPARING MODEL PERFORMANCES AND SELECTING THE BEST MODEL

Comparing all models on RMSE and R² in both log scale.
Tree-based model (XGBoost) is expected to outperform linear models
as they capture non-linear feature interactions without explicit transformation.

Comparing rmse scores

In [ ]:
rmse_scores={'Linear':np.round(rmse1,7),'Linear_with_PCA':np.round(demo_rmse1,7),'Ridge':np.round(rmse2,8),'PolynomialRegression':np.round(poly_rmse1,8),'XgBoost':np.round(rmse3,8),'XGB_with_PCA':np.round(demo_rmse3,7)}
rmses=pd.DataFrame(rmse_scores.items(),columns=['Model','RMSE'])
rmses=rmses.sort_values(by='RMSE')
rmses

Comparing r2 scores

In [ ]:
r2_scores={'Linear':np.round(r21,7),'Ridge':np.round(r22,7),'Linear_with_PCA':np.round(demo_r21,7),'PolynomialRegression':np.round(poly_r21,8),'XgBoost':np.round(r23,7),'XGB_with_PCA':np.round(demo_r23,7)}
r2s=pd.DataFrame(r2_scores.items(),columns=['Model','R2'])
r2s=r2s.sort_values(by='R2')
r2s

In [ ]:
print(rmses.iloc[0,:])

In [ ]:
print(r2s.iloc[-1,:])

## Conclusion
- Best model: **XGBoost** with R² = **0.91939** and RMSE =**$ 26061.9**
- Key drivers: Overall Quality, Kitchen and Exterior Qualities.
- Future Work: SHAP values can be added for better depiction of feature relations.